# Unit 1: Design review

**Nothing in this notebook calls a model.** It is all static analysis, so it runs
offline, costs nothing, and works in CI.

That is itself a small lesson: a large share of the highest value work on an agent
happens before any inference is bought.

## Setup

In [ ]:
from cse476.design import AgentSpec, audit_guards, audit_tools, report
from cse476.architectures import TOOL_SCHEMA, REGISTRY

print("linter loaded, no client needed")

## 1. Audit the code from Lecture 2

This should come back clean. If it does not, the linter is wrong or the code is,
and finding out which is the exercise.

In [ ]:
print(report(audit_tools(TOOL_SCHEMA, REGISTRY), "architectures.py"))

## 2. Now audit something broken

Every fault below is one I have seen in a submitted practical.

In [ ]:
bad_schema = [
    {
        "type": "function",
        "function": {
            "name": "check_and_book_room",          # two jobs in one name
            "description": "Books rooms.",          # far too thin
            "parameters": {
                "type": "object",
                "properties": {
                    "hotel": {"type": "string"},    # no description
                    "currency": {"type": "string", "description": "Currency."},
                },
                # no required list
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_weather",                  # not in the registry at all
            "description": "Current weather for a city in degrees celsius.",
            "parameters": {"type": "object",
                           "properties": {"city": {"type": "string", "description": "City name."}},
                           "required": ["city"]},
        },
    },
]


def check_and_book_room(hotel: str, date: str) -> dict:
    return {}


bad_registry = {
    "check_and_book_room": check_and_book_room,
    "orphan_tool": lambda: "never reachable",       # in registry, not in schema
}

print(report(audit_tools(bad_schema, bad_registry), "a broken agent"))

### Read the errors first

The `currency` one is worth pausing on. The schema tells the model it may send a
`currency` argument. The function does not accept one. So the moment the model uses
it, your loop raises `TypeError` inside a tool call.

**That is a runtime crash caught by reading the code.** No model call, no waiting for
it to happen in front of a user.

## 3. Audit the guards

The blunt one. Every check here corresponds to something that has already cost
somebody in this course either money or a wrong answer.

In [ ]:
print(report(audit_guards(), "an agent with no guards"))
print()
print(report(
    audit_guards(max_steps=6, has_whitelist=True,
                 has_no_progress_check=True, trims_transcript=True),
    "a fully guarded agent",
))

## 4. Write the spec before the code

Every blank field is a decision you will make anyway: later, accidentally, in the
middle of debugging something else.

Start with an empty one and watch what it demands.

In [ ]:
empty = AgentSpec(name="my-capstone")
print(report(empty.review(), "empty spec"))
print()
print("ready to build?", empty.is_ready())

In [ ]:
spec = AgentSpec(
    name="hotel-assistant",
    performance_measure="Finds an available room inside the stated budget and distance preference.",
    environment="Three hotels, their rates, distances, ratings and per date availability.",
    actuators="list_hotels, get_room_availability, get_hotel_details",
    sensors="Tool results plus the conversation so far.",
    shape="agent",
    strategy="react",
    rung="utility_based",
    what_it_refuses=["Taking payment", "Medical or legal advice"],
    pinned_facts=["stated budget", "stated dates"],
    failure_behaviour="Says it cannot find a match and names exactly what it checked.",
    max_steps=6,
)

print(report(spec.review(), "hotel-assistant"))
print()
print("ready to build?", spec.is_ready())

## 5. The linter had a bug, and that is the lesson

The first version of `audit_tools` reported this against correct code:

```
[note] get_nightly_rate    Returns str rather than str.
```

**The cause.** `from __future__ import annotations` turns every annotation into a
string, so `sig.return_annotation is str` is `False` even for a function annotated
`-> str`.

**Why it is worth a cell.** A linter that nags about correct work gets switched off,
and then it catches nothing at all. That is the same principle as the no progress
detector in Lecture 4: you have to test that a check **stays quiet**, not only that
it fires.

`tests/mock_run_l5.py` now has a case for exactly this false positive. Run it and
look at scenario 7.

In [ ]:
def annotated(hotel: str) -> str:
    """A correctly written tool. The linter should say nothing at all."""
    return hotel


clean_schema = [{
    "type": "function",
    "function": {
        "name": "annotated",
        "description": "A correctly annotated tool, used here to check the linter itself.",
        "parameters": {"type": "object",
                       "properties": {"hotel": {"type": "string", "description": "Exact hotel name."}},
                       "required": ["hotel"]},
    },
}]

findings = audit_tools(clean_schema, {"annotated": annotated})
print("findings on correct code:", findings)
assert findings == [], "the linter is nagging about correct work"
print("linter stayed quiet, which is the behaviour we want")

### One more thing worth admitting

Once that false positive was fixed, the linter immediately found three real problems
in the teaching code: none of the three tools in `architectures.py` had a docstring.

**I fixed the code, not the linter.** That is the habit worth copying.

## Your turn

**1. Audit your Practical 2 agent.** Import your own `TOOL_SCHEMA` and `REGISTRY`
and run both audits.

- Fix every **error**. These are runtime crashes waiting to happen.
- Fix or **justify** every **warning**. A short description may genuinely be fine for
  an obvious tool, but you have to say why, in a comment. An unexplained warning and
  a fixed one score the same. An ignored one does not.

**2. Write your `AgentSpec`.** Fill it in until `is_ready()` returns `True`. This
becomes the design section of your capstone proposal, so do it properly once rather
than twice.

**3. Find an anti pattern in your own code.** The god tool, the prompt that grew, the
demo shaped agent, or the unbounded budget. Name which one and what you would change.

**4. Answer these two in one sentence each.** What is the most a single request to
your agent can cost you, and what does it say when it cannot answer? If you cannot,
you have anti pattern three or four, and probably both.

In [ ]:
# your audit here
